In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, KFold

Best C: 71.96856730011521
Best CV balanced accuracy: 0.7613522175286223


In [ ]:
insurance_train = pd.read_csv("data/insurance_train.csv")
insurance_test = pd.read_csv("data/insurance_test.csv")

#Feature Engineering
#Agents, entities, and products had many rare categories so to avoid high cardinality
agents_to_keep = ['agt_0004', 'agt_0001', 'agt_0005', 'agt_0002']
insurance_train['agent_id'] = np.where(insurance_train['agent_id'].isin(agents_to_keep), insurance_train['agent_id'], 'other')
insurance_test['agent_id'] = np.where(insurance_test['agent_id'].isin(agents_to_keep), insurance_test['agent_id'], 'other')


entity_to_keep = ['50b3e71e', '96d6c6df', '99ede4e4', '7b5dbb09']
insurance_train['entity_a'] = np.where(insurance_train['entity_a'].isin(entity_to_keep), insurance_train['entity_a'], 'other')
insurance_test['entity_a'] = np.where(insurance_test['entity_a'].isin(entity_to_keep), insurance_test['entity_a'], 'other')


product_to_keep = ['TripGuard Cancel', 'SecurePlan Flex', 'DriveSafe Rental Addon', 'TravelShield Basic', 'TravelShield Core']
insurance_train['product_id'] = np.where(insurance_train['product_id'].isin(product_to_keep), insurance_train['product_id'], 'other')
insurance_test['product_id'] = np.where(insurance_test['product_id'].isin(product_to_keep), insurance_test['product_id'], 'other')

#We delive the relation between claim_status and age is not in fact linear so we decided to group it by quantiles to better capture that quality
insurance_train['age_group'] = pd.qcut(insurance_train['person_age'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
insurance_test['age_group'] = pd.qcut(insurance_test['person_age'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
insurance_train.drop('person_age', axis=1, inplace=True)
insurance_test.drop('person_age', axis=1, inplace=True)

#Location had extremly many categories and even with grouping it amounted to 19. That is why instead we decided to use a cross-validation encoding method
#and convert the variable into one numerical column where each location is represented by the mean_claim rate for it
def target_encode_cv(data, categorical_col, target_col, n_splits=5):
    encoded = pd.Series(index=data.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(data):
        train_fold, val_fold = data.iloc[train_idx], data.iloc[val_idx]
        means = train_fold.groupby(categorical_col)[target_col].mean()
        encoded.iloc[val_idx] = val_fold[categorical_col].map(means)
    global_mean = data[target_col].mean()
    return encoded.fillna(global_mean)

insurance_train['location_encoded'] = target_encode_cv(
    insurance_train, categorical_col='location', target_col='claim_status'
)
location_means = insurance_train.groupby('location')['claim_status'].mean().to_dict()
insurance_test['location_encoded'] = insurance_test['location'].map(location_means).fillna(insurance_train['claim_status'].mean())

#We tried removing the entity type and entity a variables as they were relatively highly correlated but it resulted in worse RMSE
nominal_vars = ['person_gender', 'agent_id', 'product_id', 'age_group']
extra_nominal_vars = ['entity_type', 'channel', 'entity_a', 'location']
all_nominal_vars = nominal_vars + extra_nominal_vars

#Decoding categorical variables into dummies
insurance_train_encoded = pd.get_dummies(insurance_train, columns=all_nominal_vars, drop_first=True, dtype=int)
insurance_test_encoded = pd.get_dummies(insurance_test, columns=all_nominal_vars, drop_first=True, dtype=int)

#The line below is to ensure that the test and train samples are the same as with this type of encoding as we did with location, mismatches may happen
insurance_test_encoded = insurance_test_encoded.reindex(columns=insurance_train_encoded.drop(columns='claim_status').columns, fill_value=0)

X = insurance_train_encoded.drop(columns="claim_status")
y = insurance_train_encoded["claim_status"]

# Now we build a pipeline for cross-validation using the C variable responsoble for the strengh of regularization. We choose logistic regrestion because 
# it scored the best out of the models we tested based on RMSE, and it can deal with the imbalance in the dataset (low frequency of success) by using the
# class_weight = balanced argument
C_values = np.logspace(3, -4, 50)

logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),         
    ('model', LogisticRegression(max_iter=5000,class_weight='balanced')) 
])

cv5 = KFold(n_splits=5, shuffle=True, random_state=123)

param_grid = {'model__C': C_values}

logreg_search = GridSearchCV(
    estimator=logreg_pipeline,
    param_grid=param_grid,
    scoring='balanced_accuracy',
    cv=cv5,
    n_jobs=-1                     
)

logreg_search.fit(X, y)

best_pipeline = logreg_search.best_estimator_

print("Best C:", logreg_search.best_params_['model__C'])
print("Best CV balanced accuracy:", logreg_search.best_score_)

y_test_final_pred = best_pipeline.predict(insurance_test_encoded)

prediction_output = pd.DataFrame({
    "id": range(len(y_test_final_pred)),
    "claim_status_predicted": y_test_final_pred
})

prediction_output.to_csv("outputs/final_logit_predictions.csv", index=False)